# Working with files: text, CSV and JSON

**Goal:** read data from files and write results back, using only Python's standard library. (In later weeks pandas will do much of this for you, but it helps to know what happens underneath, and APIs and config files use exactly these formats.)

This folder contains `sales_raw.csv` and `products.json`.

## 1. Where am I? Paths with `pathlib`

In [ ]:
from pathlib import Path

print(Path.cwd())                          # current working directory
print(Path("sales_raw.csv").exists())      # is the file next to this notebook?

## 2. Writing and reading text

Always use `with`: it closes the file for you, even if an error happens. Always give an `encoding` (Swedish letters like å, ä, ö need `utf-8`).

In [ ]:
with open("hello.txt", "w", encoding="utf-8") as f:
    f.write("Hej från Python!\n")
    f.write("Second line\n")

with open("hello.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)

In [ ]:
# Read line by line (memory friendly for big files)
with open("hello.txt", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

In [ ]:
# "a" appends instead of overwriting
with open("hello.txt", "a", encoding="utf-8") as f:
    f.write("Third line\n")

print(Path("hello.txt").read_text(encoding="utf-8"))

## 3. The file might not exist

In [ ]:
def read_text(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"No such file: {path}")
        return None

print(read_text("hello.txt")[:20])
print(read_text("does_not_exist.txt"))

## 4. CSV files

The `csv` module gives you each row as a list (`csv.reader`) or, much nicer, as a **dict** (`csv.DictReader`). That is the *list of dicts* shape from week 1.

In [ ]:
import csv

with open("sales_raw.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    sales = list(reader)

print(len(sales), "rows")
sales[0]

**Everything read from a CSV is a string!** Look at the types:

In [ ]:
first = sales[0]
for key, value in first.items():
    print(f"{key:12} {value:15} {type(value)}")

In [ ]:
# So we convert what we need
total = 0
for row in sales[:5]:
    total += int(row["quantity"]) * float(row["unit_price"])
print(total)

Writing a CSV with `DictWriter`:

In [ ]:
Path("output").mkdir(exist_ok=True)

electronics = []
for row in sales:
    if row["category"] == "Electronics":
        electronics.append(row)

with open("output/electronics.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=sales[0].keys())
    writer.writeheader()
    writer.writerows(electronics)

print(len(electronics), "rows written")

## 5. JSON files

JSON is the format of web APIs and config files. It maps directly to Python: object = `dict`, array = `list`, string, number, `true`/`false` = `True`/`False`, `null` = `None`.

In [ ]:
import json

with open("products.json", encoding="utf-8") as f:
    products = json.load(f)

print(type(products))
products[0]

In [ ]:
# json.dumps / json.loads work with strings instead of files
as_text = json.dumps(products[0])
print(as_text, type(as_text))
back = json.loads(as_text)
print(back, type(back))

In [ ]:
categories = []
for r in sales:
    if r["category"] not in categories:
        categories.append(r["category"])

summary = {"rows": len(sales), "categories": categories, "customers": ["Åsa", "Örjan"]}

with open("output/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)   # ensure_ascii=False keeps å ä ö readable

print(Path("output/summary.json").read_text(encoding="utf-8"))

**Not everything can be saved as JSON.** Dates, for example:

In [ ]:
# This cell raises an error on purpose
from datetime import date

json.dumps({"day": date(2025, 1, 5)})

In [ ]:
# Fix 1: convert to text yourself. Fix 2: default=str
print(json.dumps({"day": date(2025, 1, 5)}, default=str))

## 6. Putting it together: CSV to JSON

Read the CSV, keep only valid rows (using what we learned about errors), write them out as JSON.

*Note: this quick version only catches rows where the numbers cannot be converted, so it keeps 18 of 20 rows. It does not notice the empty customer or the bad date. The lab asks you to validate properly.*

In [ ]:
clean = []
for row in sales:
    try:
        clean.append({
            "order_id": int(row["order_id"]),
            "customer": row["customer"],
            "revenue": int(row["quantity"]) * float(row["unit_price"]),
        })
    except ValueError:
        pass   # bad row: skip (the lab will teach you to record it instead)

with open("output/clean_sales.json", "w", encoding="utf-8") as f:
    json.dump(clean, f, indent=2, ensure_ascii=False)

print(len(clean), "of", len(sales), "rows were valid")

## Mini exercise

Read `products.json` and write a CSV `output/products.csv` with the columns `name`, `category`, `unit_price`, sorted by price (highest first).

In [ ]:
# your code here